# Lab 8: Debugging

**Before you start: go to Runtime → Change runtime type and select GPU.**

At some point in your project, something will not work. The model will 
not learn, the loss will do something unexpected, training will crash, 
or results will be much worse than they should be.

When this happens, the difference between spending 10 minutes and 
spending 3 days finding the problem is almost always whether you have 
a systematic approach.

This lab walks through five real failure modes — each one demonstrated 
with broken code, diagnosed step by step, and then fixed. Every bug 
shown here is something that has appeared in real student projects.

**Failure modes covered:**
1. Loss not decreasing
2. Overfitting
3. Data pipeline bugs
4. Vanishing and exploding gradients
5. GPU memory errors

**Session structure:**
- First 45 minutes: work through this notebook
- Remaining time: apply the diagnostic framework to your own project. 
  Bring whatever is not working and work through it with a TA.

**The golden rule of debugging:** change one thing at a time. 
If you change three things at once, you cannot tell which one fixed it.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import zipfile, gdown
from pathlib import Path
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load dataset for use throughout the lab
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
data_dir = Path('data')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Dataset ready. Classes: {CLASS_NAMES}')

---
## Failure Mode 1: Loss not decreasing

**Symptom:** You run training and the loss barely moves, or moves very 
slowly, or oscillates without a clear downward trend.

**The code below has a bug that prevents learning. Run it and observe 
the symptom before reading the diagnosis.**

In [ ]:
# ============================================================
# BROKEN CODE — run this and observe the symptom
# ============================================================

def broken_training_loop(model, train_dl, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    losses    = []

    for epoch in range(epochs):
        epoch_loss = 0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)

            # BUG IS SOMEWHERE IN THESE LINES
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            # optimizer.zero_grad()  <-- moved to AFTER step

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_dl)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}: loss = {avg_loss:.4f}')

    return losses

model = models.resnet18(weights=None)   # no pretrained weights for this demo
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

broken_losses = broken_training_loop(model, train_dl, epochs=5)

plt.plot(broken_losses, 'r-o', label='Broken (gradients accumulate)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss not decreasing properly')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Diagnosis:** The `optimizer.zero_grad()` call is missing before 
`loss.backward()`. Without it, gradients from previous batches 
accumulate in the parameter tensors. By the second batch, gradients 
are twice as large as they should be; by the tenth batch, they are 
ten times too large. The effective learning rate explodes, causing 
erratic parameter updates.

**This is the single most common training bug.** It is easy to miss 
because the code still runs — it just does not learn properly.

**How to spot it:** Loss oscillates or is much higher than expected. 
The loss at epoch 1 is similar to epoch 5.

**The fix:**

In [ ]:
# ============================================================
# FIXED CODE
# ============================================================

def fixed_training_loop(model, train_dl, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    losses    = []

    for epoch in range(epochs):
        epoch_loss = 0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()   # FIX: clear gradients BEFORE backward
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_dl)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}: loss = {avg_loss:.4f}')

    return losses

torch.manual_seed(42)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

fixed_losses = fixed_training_loop(model, train_dl, epochs=5)

plt.plot(broken_losses, 'r--o', label='Broken')
plt.plot(fixed_losses,  'g-o',  label='Fixed')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Loss curves: broken vs fixed')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print('Fix: call optimizer.zero_grad() at the START of each batch, before loss.backward()')

**Other common causes of loss not decreasing:**

| Cause | How to spot it | Fix |
|---|---|---|
| Learning rate too high | Loss oscillates wildly | Reduce lr by 10x |
| Learning rate too low | Loss decreases but very slowly | Increase lr by 10x |
| Wrong loss function | Loss decreases but metric does not improve | Match loss to task |
| Labels and images mismatched | Model never improves despite correct code | Visualise batches |
| Model in eval mode during training | Loss does not decrease | Call `model.train()` before training loop |

---
## Failure Mode 2: Overfitting

**Symptom:** Training accuracy is high (>95%), but validation accuracy 
is much lower and stops improving early. The model has memorised 
the training set instead of learning to generalise.

In [ ]:
# Demonstrate overfitting by training on a tiny subset with no regularisation
from torch.utils.data import Subset

# Use only 60 training images (20 per class) — too few to generalise
tiny_indices = []
for cls in range(NUM_CLASSES):
    cls_indices = [i for i, (_, l) in enumerate(train_ds.samples) if l == cls][:20]
    tiny_indices.extend(cls_indices)

tiny_ds = Subset(train_ds, tiny_indices)
tiny_dl = DataLoader(tiny_ds, batch_size=16, shuffle=True)

def train_and_track(model, train_dl, val_dl, epochs=20, lr=1e-3, wd=0.0):
    """Train and return per-epoch train and val accuracy."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        model.train()
        t_corr = t_tot = 0
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            t_corr += (out.argmax(1) == lbls).sum().item()
            t_tot  += imgs.size(0)

        model.eval()
        v_corr = v_tot = 0
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                v_corr += (out.argmax(1) == lbls).sum().item()
                v_tot  += imgs.size(0)

        train_accs.append(t_corr / t_tot)
        val_accs.append(v_corr / v_tot)

    return train_accs, val_accs


# Overfit: no regularisation, tiny dataset
torch.manual_seed(42)
overfit_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
overfit_model.fc = nn.Linear(overfit_model.fc.in_features, NUM_CLASSES)
overfit_model = overfit_model.to(device)

print('Training on tiny dataset (no regularisation)...')
train_accs_overfit, val_accs_overfit = train_and_track(
    overfit_model, tiny_dl, val_dl, epochs=20, lr=1e-3, wd=0.0
)

epochs = range(1, 21)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_accs_overfit, 'b-',  label='Train accuracy')
ax.plot(epochs, val_accs_overfit,   'r--', label='Val accuracy')
ax.fill_between(epochs, val_accs_overfit, train_accs_overfit,
                alpha=0.1, color='red', label='Generalisation gap')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: train accuracy >> val accuracy')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

gap = max(train_accs_overfit) - val_accs_overfit[np.argmax(train_accs_overfit)]
print(f'Generalisation gap: {gap:.3f} ({gap*100:.1f}%)')

In [ ]:
# Fix: add regularisation and data augmentation
stronger_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tiny_ds_aug = Subset(
    datasets.ImageFolder(data_dir / 'train', transform=stronger_transform),
    tiny_indices
)
tiny_dl_aug = DataLoader(tiny_ds_aug, batch_size=16, shuffle=True)

torch.manual_seed(42)
regularised_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
regularised_model.fc = nn.Linear(regularised_model.fc.in_features, NUM_CLASSES)
regularised_model = regularised_model.to(device)

print('Training with regularisation (augmentation + weight decay + dropout head)...')
train_accs_reg, val_accs_reg = train_and_track(
    regularised_model, tiny_dl_aug, val_dl,
    epochs=20, lr=1e-3, wd=1e-3   # weight decay added
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_accs_overfit, 'b--', alpha=0.4, label='Train (no reg)')
ax.plot(epochs, val_accs_overfit,   'r--', alpha=0.4, label='Val (no reg)')
ax.plot(epochs, train_accs_reg,     'b-',  label='Train (with reg)')
ax.plot(epochs, val_accs_reg,       'r-',  label='Val (with reg)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Effect of regularisation on overfitting')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Regularisation closes the generalisation gap.')
print('Options: data augmentation, weight decay, dropout, early stopping.')

---
## Failure Mode 3: Data pipeline bugs

**Symptom:** The model trains without errors, loss decreases, but 
results are much worse than expected — or worse than random chance.

Data pipeline bugs are the most insidious category because the code 
runs without crashing. The model learns *something*, just not what you intended.

**Three common data bugs demonstrated below:**

In [ ]:
def show_batch(loader, class_names, title, n=8):
    """Visualise a batch from a dataloader — your first line of defence."""
    imgs, labels = next(iter(loader))

    def denorm(t):
        mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
        std  = torch.tensor(IMAGENET_STD).view(3,1,1)
        return torch.clamp(t * std + mean, 0, 1)

    fig, axes = plt.subplots(1, min(n, len(imgs)), figsize=(2.5 * min(n, len(imgs)), 3))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, img, lbl in zip(axes, imgs[:n], labels[:n]):
        ax.imshow(denorm(img).permute(1,2,0).numpy())
        ax.set_title(class_names[lbl], fontsize=8)
        ax.axis('off')
    plt.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()

# Bug 1: Wrong normalisation
# Common mistake: using mean/std values for one dataset on another,
# or normalising twice.
wrong_norm_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),  # BUG: normalised twice
])

correct_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

wrong_ds  = datasets.ImageFolder(data_dir / 'val', transform=wrong_norm_transform)
correct_ds = datasets.ImageFolder(data_dir / 'val', transform=correct_transform)

wrong_dl   = DataLoader(wrong_ds,   batch_size=8, shuffle=False)
correct_dl = DataLoader(correct_ds, batch_size=8, shuffle=False)

print('Bug 1: Double normalisation')
wrong_imgs, _   = next(iter(wrong_dl))
correct_imgs, _ = next(iter(correct_dl))

print(f'  Correct tensor range: [{correct_imgs.min():.2f}, {correct_imgs.max():.2f}]')
print(f'  Wrong tensor range:   [{wrong_imgs.min():.2f}, {wrong_imgs.max():.2f}]')
print('  If range is very large, normalisation has been applied multiple times.')
print('  Check your transform pipeline — normalisation should appear exactly once.')

In [ ]:
# Bug 2: Labels shuffled — images and labels do not correspond
# This happens when you build a dataset manually from separate lists
# and accidentally shuffle one but not the other.

from glob import glob
import random

cat_paths  = sorted(glob(str(data_dir / 'val/cat/*.jpg')))
dog_paths  = sorted(glob(str(data_dir / 'val/dog/*.jpg')))
horse_paths = sorted(glob(str(data_dir / 'val/horse/*.jpg')))

all_paths  = cat_paths + dog_paths + horse_paths
all_labels = [0] * len(cat_paths) + [1] * len(dog_paths) + [2] * len(horse_paths)

# BUG: shuffle paths but not labels
random.seed(42)
shuffled_paths = all_paths.copy()
random.shuffle(shuffled_paths)   # labels stay in original order!

print('Bug 2: Shuffled images, unshuffled labels')
print(f'  First 5 labels: {all_labels[:5]}  (0=cat)')
print(f'  First 5 paths (shuffled):')
for p in shuffled_paths[:5]:
    print(f'    {Path(p).parent.name} → labelled as {CLASS_NAMES[all_labels[0]]}')
print('  Images and labels no longer correspond!')
print('  FIX: shuffle (path, label) pairs together using zip + list + shuffle')
print()

# Correct approach
paired = list(zip(all_paths, all_labels))
random.shuffle(paired)
correct_paths, correct_labels = zip(*paired)
print('  Correct: shuffle pairs')
for p, l in zip(list(correct_paths)[:3], list(correct_labels)[:3]):
    print(f'    {Path(p).parent.name} → labelled as {CLASS_NAMES[l]}  ✓')

In [ ]:
# Bug 3: Applying val transforms to training data (no augmentation)
# This is subtle — the model trains fine but misses regularisation benefits.
# Worse: applying train transforms (with augmentation) to validation data
# gives a noisy, unreliable validation metric.

# Correct: different transforms for train and val
correct_train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
correct_val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)

# Bug: same transform used for both (train transform on val = noisy val metric)
buggy_val_ds = datasets.ImageFolder(data_dir / 'val', transform=train_transform)

print('Bug 3: Augmentation applied to validation data')
print('  Training transform:   ', train_transform.transforms)
print()
print('  When applied to validation data:')
print('  - Random flips mean the same image gives different predictions each run')
print('  - Validation accuracy varies between runs — hard to compare experiments')
print('  - True val accuracy is masked by augmentation noise')
print()
print('  FIX: always use deterministic transforms (Resize + ToTensor + Normalize)')
print('  on validation and test data. Augmentation is ONLY for training.')

# Visualise the difference
show_batch(
    DataLoader(correct_val_ds, batch_size=8, shuffle=False),
    CLASS_NAMES, 'Correct: deterministic val transforms'
)
show_batch(
    DataLoader(buggy_val_ds, batch_size=8, shuffle=False),
    CLASS_NAMES, 'Bug: augmentation applied to val data (run again — will differ)'
)

**The most important debugging tool for data pipelines is `show_batch`.**
Always visualise a few batches before training. It takes 30 seconds and 
catches most data bugs immediately.

---
## Failure Mode 4: Vanishing and exploding gradients

**Symptom (vanishing):** Loss decreases very slowly or not at all, 
even with a correct learning rate. Early layers of the network do not learn.

**Symptom (exploding):** Loss suddenly jumps to NaN or a very large 
value after some training steps. Weights become NaN.

These problems are most common when training deep networks from scratch, 
using RNNs on long sequences, or using very high learning rates.

In [ ]:
def check_gradients(model, loader, criterion, n_batches=3):
    """
    Run a few forward+backward passes and inspect gradient magnitudes.
    Call this BEFORE a full training run to catch gradient problems early.
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for i, (imgs, lbls) in enumerate(loader):
        if i >= n_batches:
            break
        optimizer.zero_grad()
        loss = criterion(model(imgs.to(device)), lbls.to(device))
        loss.backward()

    # Collect gradient norms per layer
    layer_names, grad_norms = [], []
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.norm().item()
            layer_names.append(name)
            grad_norms.append(grad_norm)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(range(len(grad_norms)), grad_norms, 'b-', alpha=0.7)
    ax.set_xticks(range(0, len(layer_names), max(1, len(layer_names)//10)))
    ax.set_xticklabels(
        [layer_names[i] for i in range(0, len(layer_names), max(1, len(layer_names)//10))],
        rotation=45, ha='right', fontsize=7
    )
    ax.set_ylabel('Gradient norm (log scale)')
    ax.set_title('Gradient norms across layers')
    ax.axhline(1e-6, color='r', linestyle='--', label='Vanishing threshold')
    ax.axhline(1e2,  color='orange', linestyle='--', label='Exploding threshold')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f'Min gradient norm: {min(grad_norms):.2e}')
    print(f'Max gradient norm: {max(grad_norms):.2e}')
    print(f'Any NaN gradients: {any(np.isnan(g) for g in grad_norms)}')

    if any(g < 1e-6 for g in grad_norms):
        print('WARNING: Some gradients near zero — possible vanishing gradient problem.')
    if any(g > 1e2 for g in grad_norms):
        print('WARNING: Some gradients very large — possible exploding gradient problem.')


# Check gradient health on a pretrained ResNet-18
criterion = nn.CrossEntropyLoss()
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

print('Gradient check on pretrained ResNet-18:')
check_gradients(model, train_dl, criterion)

In [ ]:
# Demonstrate exploding gradients with a very high learning rate
def demonstrate_explosion(model, loader, bad_lr=10.0, steps=20):
    """Show what exploding gradients look like in practice."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=bad_lr)   # absurdly high lr
    losses = []

    for step, (imgs, lbls) in enumerate(loader):
        if step >= steps:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        losses.append(loss.item())
        optimizer.step()

    return losses


torch.manual_seed(0)
m_explode = models.resnet18(weights=None)
m_explode.fc = nn.Linear(m_explode.fc.in_features, NUM_CLASSES)

bad_losses  = demonstrate_explosion(m_explode, train_dl, bad_lr=10.0)

torch.manual_seed(0)
m_good = models.resnet18(weights=None)
m_good.fc = nn.Linear(m_good.fc.in_features, NUM_CLASSES)
good_losses = demonstrate_explosion(m_good, train_dl, bad_lr=1e-3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(bad_losses,  'r-', label='lr=10.0 (exploding)')
axes[0].set_title('Exploding gradients (lr too high)')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(good_losses, 'g-', label='lr=1e-3 (stable)')
axes[1].set_title('Stable training (lr correct)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Fixes for exploding gradients:')
print('  1. Reduce learning rate')
print('  2. Gradient clipping: torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)')
print('  3. Use batch normalisation')
print('  4. Use a well-initialised pretrained model instead of random init')

---
## Failure Mode 5: GPU memory errors

**Symptom:** `RuntimeError: CUDA out of memory` — training crashes, 
sometimes after running fine for several batches.

**Why it happens:** GPU memory is finite. Each forward pass stores 
intermediate activations for backpropagation. Large batches, large 
models, or large input images can exceed available memory.

In [ ]:
def check_gpu_memory():
    """Report current GPU memory usage."""
    if device.type != 'cuda':
        print('No GPU available — skipping memory check.')
        return
    allocated = torch.cuda.memory_allocated(device) / 1e9
    reserved  = torch.cuda.memory_reserved(device)  / 1e9
    total     = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f'GPU memory: {allocated:.2f} GB allocated / '
          f'{reserved:.2f} GB reserved / {total:.2f} GB total')


# Strategies for reducing GPU memory usage
print('GPU memory management strategies:')
print()

print('1. Reduce batch size')
print('   Memory scales linearly with batch size.')
print('   If you get OOM with batch_size=64, try 32 or 16.')
print()

print('2. Use torch.no_grad() during validation')
print('   Validation does not need gradients — no_grad() cuts memory in half.')
print()

print('3. Use gradient checkpointing for very deep models')
print('   Trades compute for memory: recomputes activations instead of storing them.')
print()

print('4. Use mixed precision training')
print('   float16 uses half the memory of float32, with negligible accuracy loss.')
print()

print('5. Reduce image resolution')
print('   Memory scales quadratically with image size.')
print('   224x224 uses 4x less memory than 448x448.')
print()

check_gpu_memory()

# Example: mixed precision training
print('\nExample: mixed precision training with torch.cuda.amp')

In [ ]:
# Mixed precision training — halves GPU memory usage
# Drop-in replacement for the standard training loop

def train_mixed_precision(model, train_dl, epochs=2):
    """
    Training loop with automatic mixed precision (AMP).
    Uses float16 for forward pass, float32 for gradient updates.
    Typically 1.5-2x faster and uses ~half the GPU memory.
    """
    if device.type != 'cuda':
        print('Mixed precision requires a GPU — skipping.')
        return

    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scaler    = torch.cuda.amp.GradScaler()   # handles float16 stability

    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()

            # Forward pass in float16
            with torch.cuda.amp.autocast():
                out  = model(imgs)
                loss = criterion(out, lbls)

            # Backward pass: scaler prevents underflow in float16 gradients
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        check_gpu_memory()
        print(f'Epoch {epoch} complete')


torch.manual_seed(42)
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

print('Before training:')
check_gpu_memory()
print()
train_mixed_precision(model, train_dl, epochs=2)

---
## The Debugging Checklist

When something is not working, work through this checklist in order. 
Stop as soon as you find the problem — do not fix multiple things at once.

### Step 1: Overfit a single batch
Before anything else, verify that your model *can* learn:

In [ ]:
def overfit_single_batch(model, loader, steps=50):
    """
    THE most important debugging test.

    Take one batch and train on it repeatedly. A correctly implemented model
    should reach near-zero loss on a single batch within ~50 steps.

    If it cannot do this, there is a fundamental bug in your model or loss.
    There is no point debugging anything else until this test passes.
    """
    model  = model.to(device)
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Get one fixed batch
    imgs, lbls = next(iter(loader))
    imgs, lbls = imgs.to(device), lbls.to(device)

    losses = []
    for step in range(steps):
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Overfit single batch test')
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    final_acc = (model(imgs).argmax(1) == lbls).float().mean().item()
    print(f'Final loss:     {losses[-1]:.4f} (should be near 0)')
    print(f'Final accuracy: {final_acc:.3f} (should be near 1.0)')

    if losses[-1] < 0.01 and final_acc > 0.99:
        print('✓ PASS: model can overfit a single batch. Pipeline is correct.')
    else:
        print('✗ FAIL: model cannot overfit a single batch.')
        print('  There is a fundamental bug in your model, loss, or training loop.')
        print('  Fix this before doing anything else.')


torch.manual_seed(42)
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

overfit_single_batch(model, train_dl)

In [ ]:
# Full debugging checklist — print and keep nearby
checklist = """
DEBUGGING CHECKLIST
===================
Work through this in order. Stop at the first item that fails.

□ 1. Can the model overfit a single batch?
     Run overfit_single_batch(). Loss should reach near 0 in 50 steps.
     If not: bug in model architecture or loss function.

□ 2. Is the data pipeline correct?
     Visualise batches with show_batch().
     Check: images look right, labels are correct, normalisation is sensible.

□ 3. Is optimizer.zero_grad() called before every backward pass?
     Missing zero_grad is the single most common training bug.

□ 4. Is model.train() called before training, model.eval() before validation?
     Forgetting model.eval() makes dropout active during validation.

□ 5. Are the initial loss values as expected?
     For cross-entropy with N classes, initial loss ≈ log(N).
     3 classes → ≈ 1.10.  10 classes → ≈ 2.30.
     If much higher, check normalisation or model output range.

□ 6. Are gradients healthy?
     Run check_gradients(). Norms should be in range [1e-6, 1e2].
     NaN gradients: reduce lr or check for 0-division in loss.

□ 7. Is the learning rate in a reasonable range?
     Start with 1e-3 for Adam, 1e-1 for SGD.
     If loss does not decrease: try 10x higher.
     If loss oscillates: try 10x lower.

□ 8. Is validation showing overfitting?
     Train acc >> val acc: add augmentation, weight decay, or dropout.
     Both low: increase model capacity or train longer.

□ 9. Are there class imbalance issues?
     Check per-class accuracy. One class at 100%, others at random chance
     → model is predicting the majority class only.

□ 10. Running out of GPU memory?
      Reduce batch size. Use torch.no_grad() in eval loop.
      Consider mixed precision (torch.cuda.amp).
"""
print(checklist)

## Now: apply the framework to your project

For the rest of the session, bring whatever is not working in your 
project and work through it with a TA using the checklist above.

**If everything is working:** run `overfit_single_batch` and 
`check_gradients` on your model anyway. Catching a latent problem 
now is much better than discovering it later.

**If you are stuck:** start with step 1 of the checklist, even if 
you think the problem is something else. The single-batch overfit 
test has a way of revealing issues that seem unrelated.